# 04 · PyTorch Fundamentals

In plain English, PyTorch is the toolbox that almost every modern language model is built and trained with. Before we fine-tune a real Hugging Face model, we need to be comfortable with its building blocks: **tensors** (number grids, like NumPy arrays but supercharged), the **automatic gradient** machinery that does the calculus for us, and the small set of objects — models, loss functions, optimizers — that every training script snaps together like LEGO bricks.

You already did some of this *by hand* in notebook 03 (computing a gradient and nudging a parameter). Here you'll meet the tools that do all of that for you, automatically, on any model — no matter how big.

## What you'll learn

- **Tensors:** creating them (`torch.tensor`, `zeros`, `ones`, `randn`, `arange`), reading their `shape` and `dtype`, indexing/slicing, reshaping with `view`/`reshape`, and converting to/from NumPy.
- **Tensor math:** elementwise operations, matrix multiply (`@` / `torch.matmul`), **broadcasting**, and reductions (`sum`, `mean`, `argmax`).
- **Devices:** detecting `cuda` / `mps` / `cpu` and moving tensors with `.to(device)` — written so it runs anywhere, falling back to CPU.
- **Autograd:** turning on `requires_grad`, building an expression, calling `.backward()`, and reading `.grad` — the automatic version of the hand-calculus from notebook 03.
- **`nn.Module` and `nn.Linear`:** defining a tiny model class with `__init__` + `forward`, inspecting its parameters, and calling it like a function.
- **Loss functions:** `nn.MSELoss` and `nn.CrossEntropyLoss`, and exactly what each one expects as input.
- **Optimizers:** `torch.optim.SGD` and `Adam`, and the `zero_grad → backward → step` rhythm.
- **Hands-on:** a tiny linear-regression example where you watch the loss go down.

## Why this matters for fine-tuning

Fine-tuning is *not* magic — under the hood it's pure PyTorch:

- **Every Hugging Face model is an `nn.Module`.** When you write `model = AutoModel.from_pretrained(...)`, you get back exactly the kind of object you'll build in this notebook — just much bigger.
- **Training uses these exact pieces.** The fine-tuning loop creates an optimizer (`AdamW`, a cousin of `Adam`), computes a loss (often `CrossEntropyLoss`), calls `loss.backward()`, and steps the optimizer. Same four lines you'll learn here.
- **Most real bugs are shape errors.** "Expected size 768 but got 512" is the most common error you'll hit. Understanding tensor `shape` and broadcasting is what lets you *read the error and fix it* instead of guessing.

Get comfortable here and the real fine-tuning notebooks will feel like assembling familiar parts.

## Setup

Run the cell below once. The `%pip install` line is **commented out** — uncomment it if you're on Google Colab or a fresh environment without PyTorch. The plain CPU build of PyTorch is enough for everything in this notebook; you do **not** need a GPU.

In [ ]:
# Uncomment the next line on Colab or a fresh environment:
# %pip install torch

import torch
import torch.nn as nn   # the "neural network" toolbox: layers, losses, models

# A reproducible random seed so your numbers match across re-runs.
torch.manual_seed(0)

print("PyTorch version:", torch.__version__)   # -> e.g. 2.x.x (exact version varies)
print("Imports OK")                            # -> Imports OK

## 1. Creating tensors

A **tensor** is PyTorch's word for a grid of numbers. A single number is a 0-D tensor (a *scalar*), a list of numbers is 1-D (a *vector*), a table is 2-D (a *matrix*), and you can keep going. If you know NumPy arrays, you already understand tensors — they're the same idea with two superpowers added: they can run on a **GPU** and they can **track gradients** (more on both below).

Here are the everyday ways to make one.

In [ ]:
# From a Python list (PyTorch guesses the dtype from the values).
a = torch.tensor([1.0, 2.0, 3.0])
print("a:", a)                 # -> a: tensor([1., 2., 3.])

# All zeros / all ones, given a SHAPE (rows, cols).
zeros = torch.zeros(2, 3)      # 2 rows, 3 columns, all 0.0
ones  = torch.ones(2, 3)       # same shape, all 1.0
print("zeros:\n", zeros)
print("ones:\n", ones)

# Random numbers from a normal (bell-curve) distribution — how model weights start out.
r = torch.randn(2, 3)          # values vary every run unless a seed is set
print("randn:\n", r)           # -> a 2x3 grid of small positive/negative numbers

# A range of evenly spaced integers, like Python's range() but as a tensor.
seq = torch.arange(0, 6)       # 0,1,2,3,4,5  (stop is EXCLUSIVE)
print("arange:", seq)          # -> tensor([0, 1, 2, 3, 4, 5])

**What this does:** `torch.tensor` wraps existing numbers; `zeros`/`ones`/`randn` create new tensors *from a shape*; `arange` makes a sequence. `randn` is special — model weights are initialized with random numbers like these, then training reshapes them into something useful.

### ✏️ Exercise

**Task:** Create a tensor named `grid` that is 3 rows by 4 columns and filled entirely with the number `7`. Print its values.

**Hint:** There's no `torch.sevens`, but `torch.ones(...)` gives you all 1s — what happens if you multiply that by 7? (Or look up `torch.full`.)

In [ ]:
# Your turn! Try it before peeking at the solution below.


# ---- Sample solution ----
grid = torch.ones(3, 4) * 7        # all-ones grid, scaled to 7
# Equivalent: grid = torch.full((3, 4), 7.0)
print(grid)
# -> a 3x4 tensor where every value is 7.

## 2. Shape, dtype, indexing, and slicing

Two questions you'll ask a tensor a thousand times: **"what shape are you?"** and **"what type of numbers do you hold?"** The answers are `.shape` and `.dtype`. Getting these right is 90% of debugging.

Indexing and slicing work just like Python lists and NumPy arrays.

In [ ]:
m = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

print("shape:", m.shape)       # -> torch.Size([2, 3])  (2 rows, 3 cols)
print("dtype:", m.dtype)       # -> torch.float32  (the default for floats)
print("how many dims:", m.ndim)  # -> 2

# Indexing: [row, col]. Counting starts at 0.
print("m[0, 0]:", m[0, 0])     # -> tensor(1.)  (top-left)
print("m[1, 2]:", m[1, 2])     # -> tensor(6.)  (bottom-right)

# Slicing: [start:stop] per dimension. ":" means "all of this dimension".
print("first row:", m[0])      # -> tensor([1., 2., 3.])
print("first column:", m[:, 0])  # -> tensor([1., 4.])  (all rows, column 0)

# dtype matters: integers vs floats behave differently.
ints = torch.tensor([1, 2, 3])         # no decimals -> int64
print("ints dtype:", ints.dtype)        # -> torch.int64
print("as floats:", ints.float().dtype) # -> torch.float32  (.float() converts)

**What this does:** `.shape` tells you the size along each dimension, `.dtype` tells you the number type. A pull-your-hair-out class of bugs comes from a tensor being `int64` when a layer wanted `float32`, or being shape `(512,)` when something wanted `(768,)`. Now you know how to check.

### ✏️ Exercise

**Task:** From the tensor `m` above, extract the **second row** as its own tensor, and separately print the value in the **last column of the first row**.

**Hint:** Rows are the first index. "Second" means index `1`. For the last column you can use index `-1` (negative indexing counts from the end), just like Python lists.

In [ ]:
# Your turn!


# ---- Sample solution ----
second_row = m[1]              # -> tensor([4., 5., 6.])
last_of_first = m[0, -1]       # -> tensor(3.)
print("second row:", second_row)
print("last col of first row:", last_of_first)

## 3. Reshaping with `view` and `reshape`

The *same* numbers can be arranged into different shapes. A flat list of 6 numbers can become a 2x3 grid or a 3x2 grid. This is constant in deep learning — you'll flatten an image into a vector, or fold a batch of token embeddings into the shape a layer expects.

`view` and `reshape` do almost the same thing. Use `reshape` when in doubt — it always works; `view` is a faster version that occasionally needs the data to be laid out contiguously in memory.

In [ ]:
x = torch.arange(6)            # tensor([0, 1, 2, 3, 4, 5]), shape (6,)
print("original shape:", x.shape)   # -> torch.Size([6])

# Rearrange the same 6 numbers into 2 rows x 3 cols.
x23 = x.reshape(2, 3)
print("2x3:\n", x23)
# -> tensor([[0, 1, 2],
#            [3, 4, 5]])

# view() does the same here.
x32 = x.view(3, 2)
print("3x2:\n", x32)

# -1 means "you figure out this dimension for me."
auto = x.reshape(3, -1)        # 3 rows, columns inferred -> 2
print("reshape(3, -1) shape:", auto.shape)   # -> torch.Size([3, 2])

# The total number of elements must stay the same (here, always 6).

**What this does:** Reshaping reinterprets the *layout* of the same data without changing the values. The handy `-1` lets PyTorch compute one dimension automatically, so you only specify the dimensions you care about. The one rule: the total element count can't change (6 stays 6).

### ✏️ Exercise

**Task:** Make a tensor of the numbers 0 through 11 (twelve numbers), then reshape it into a 4-row by 3-column grid. Then reshape that into 2 rows using `-1` for the columns, and print the resulting shape.

**Hint:** Use `torch.arange(12)`. For the second reshape, `reshape(2, -1)` lets PyTorch figure out the column count.

In [ ]:
# Your turn!


# ---- Sample solution ----
t = torch.arange(12)
t43 = t.reshape(4, 3)
print("4x3:\n", t43)
t2 = t43.reshape(2, -1)        # 2 rows; columns inferred -> 6
print("2-row shape:", t2.shape)   # -> torch.Size([2, 6])

## 4. The NumPy bridge

PyTorch and NumPy are best friends. You'll often load data with NumPy (or pandas, which sits on NumPy) and then hand it to PyTorch. Two methods connect the worlds:

- `tensor.numpy()` — turn a tensor into a NumPy array.
- `torch.from_numpy(array)` — turn a NumPy array into a tensor.

In [ ]:
import numpy as np

# NumPy -> PyTorch
np_arr = np.array([1.0, 2.0, 3.0], dtype=np.float32)
t = torch.from_numpy(np_arr)
print("from NumPy:", t)        # -> tensor([1., 2., 3.])

# PyTorch -> NumPy
back = t.numpy()
print("back to NumPy:", back, type(back))   # -> [1. 2. 3.] <class 'numpy.ndarray'>

# Heads-up: on CPU these often SHARE memory. Changing one can change the other!
np_arr[0] = 99.0
print("tensor after editing the array:", t)  # -> tensor([99.,  2.,  3.])

**What this does:** The two converters let you move data between the libraries with no copying overhead. The shared-memory behavior is a classic surprise: on CPU, `from_numpy` and `.numpy()` can return objects that point at the *same* underlying numbers, so an edit in one shows up in the other. Use `.clone()` (tensor) or `.copy()` (NumPy) when you want an independent copy.

### ✏️ Exercise

**Task:** Create a NumPy array of `[10, 20, 30, 40]` (use `dtype=np.float32`), convert it to a tensor, multiply the tensor by 2, and print the result.

**Hint:** `torch.from_numpy(...)` gives you the tensor; ordinary `* 2` works on tensors just like on NumPy arrays.

In [ ]:
# Your turn!


# ---- Sample solution ----
arr = np.array([10, 20, 30, 40], dtype=np.float32)
tt = torch.from_numpy(arr)
print(tt * 2)                  # -> tensor([20., 40., 60., 80.])

## 5. Tensor math: elementwise ops and reductions

Most tensor math is **elementwise**: `a + b` adds matching positions, `a * b` multiplies matching positions, and so on. **Reductions** collapse a tensor down to fewer numbers — `sum`, `mean`, `max`, and `argmax` (the *position* of the max, which is how a classifier picks its predicted label).

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])

print("a + b:", a + b)         # -> tensor([11., 22., 33.])  (position by position)
print("a * b:", a * b)         # -> tensor([10., 40., 90.])  (NOT matrix multiply)
print("a ** 2:", a ** 2)       # -> tensor([1., 4., 9.])

# Reductions collapse many numbers into one (or a few).
print("sum:", a.sum())         # -> tensor(6.)
print("mean:", a.mean())       # -> tensor(2.)
print("max:", b.max())         # -> tensor(30.)
print("argmax:", b.argmax())   # -> tensor(2)  (index of the largest value)

# On 2-D tensors you can reduce along a specific dimension with dim=.
g = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print("sum over columns (dim=0):", g.sum(dim=0))  # -> tensor([5., 7., 9.])
print("sum over rows (dim=1):", g.sum(dim=1))     # -> tensor([6., 15.])

**What this does:** Elementwise ops keep the shape and combine matching positions. Reductions shrink the tensor: `mean` is how a loss becomes one number, and `argmax` is how a model turns a row of scores into a single predicted class. The `dim=` argument chooses *which* direction to collapse — `dim=0` walks down the rows (giving a per-column result), `dim=1` walks across the columns (giving a per-row result).

### ✏️ Exercise

**Task:** A model produced these raw scores (called *logits*) for 3 classes: `[2.0, 5.0, 1.0]`. Which class did it predict? Print the predicted class index.

**Hint:** The predicted class is the position of the largest score — that's exactly what `argmax` returns.

In [ ]:
# Your turn!


# ---- Sample solution ----
logits = torch.tensor([2.0, 5.0, 1.0])
predicted = logits.argmax()
print("predicted class index:", predicted)   # -> tensor(1)  (5.0 is largest)

## 6. Matrix multiply and broadcasting

Two more ideas power *every* neural network layer:

- **Matrix multiply** (`@` or `torch.matmul`) is the core operation a `Linear` layer performs. It is **not** the same as `*` (elementwise). The rule: an `(m, k)` matrix times a `(k, n)` matrix gives an `(m, n)` matrix — the inner sizes (`k`) must match.
- **Broadcasting** lets tensors of *different but compatible* shapes combine, by virtually stretching the smaller one. It's how you add a single bias number to a whole row, or one bias *vector* to every row of a batch.

In [ ]:
# Matrix multiply: (2x3) @ (3x2) -> (2x2). Inner 3's must match.
A = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])      # shape (2, 3)
B = torch.tensor([[1.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 1.0]])           # shape (3, 2)
print("A @ B:\n", A @ B)                 # same as torch.matmul(A, B)
print("result shape:", (A @ B).shape)    # -> torch.Size([2, 2])

# Broadcasting: add a single number to every element.
print("A + 100:\n", A + 100)

# Broadcasting: add a length-3 row to EVERY row of A (shapes (2,3) and (3,) match up).
bias = torch.tensor([10.0, 20.0, 30.0])
print("A + bias:\n", A + bias)
# -> tensor([[11., 22., 33.],
#            [14., 25., 36.]])

**What this does:** `@` performs the rows-times-columns matrix multiply at the heart of a `Linear` layer (it computes `inputs @ weights`). Broadcasting auto-aligns shapes so you can add a bias vector to a whole batch in one line. When a matrix multiply errors with "size mismatch," it's almost always because the inner dimensions don't line up — read the two shapes and check that the middle numbers match.

### ✏️ Exercise

**Task:** You have an input row `x` of shape `(1, 3)` and a weight matrix `W` of shape `(3, 2)`. Compute `x @ W` and print the result and its shape. What shape do you expect?

**Hint:** `(1, 3) @ (3, 2)` -> the inner 3's match, so the output is `(1, 2)`.

In [ ]:
# Your turn!


# ---- Sample solution ----
x = torch.tensor([[1.0, 2.0, 3.0]])      # shape (1, 3)
W = torch.tensor([[1.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 1.0]])           # shape (3, 2)
out = x @ W
print("out:", out)                       # -> tensor([[4., 5.]])
print("shape:", out.shape)               # -> torch.Size([1, 2])

## 7. Devices: CPU, CUDA, and MPS

Tensors live on a **device**. The default is the **CPU**, which works everywhere. If you have an NVIDIA GPU, that device is called `cuda`; on a modern Mac, Apple's GPU is called `mps`. GPUs make big models train far faster, but every example in this notebook runs fine on CPU.

The professional habit is to *detect* the best available device and write code that runs anywhere. Then `.to(device)` moves a tensor (or a whole model) onto it.

In [ ]:
# Pick the best device available, but always fall back to CPU.
if torch.cuda.is_available():
    device = torch.device("cuda")        # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = torch.device("mps")         # Apple Silicon GPU
else:
    device = torch.device("cpu")         # works everywhere

print("Using device:", device)           # -> cpu  (unless you have a GPU)

# Move a tensor onto the chosen device.
t = torch.randn(2, 3).to(device)
print("tensor device:", t.device)        # -> matches `device` above

# Golden rule: tensors must be on the SAME device to combine.
# A CPU tensor + a GPU tensor raises an error. .to(device) keeps them together.

**What this does:** The `if/elif/else` ladder asks PyTorch what hardware exists and picks the fastest, never crashing on a plain laptop. `.to(device)` relocates data. The most common device bug in fine-tuning is mixing a CPU tensor with a GPU model — the fix is always to `.to(device)` everything consistently.

### ✏️ Exercise

**Task:** Create a tensor of zeros with shape `(4,)`, move it to the `device` detected above, and print its `.device`.

**Hint:** `torch.zeros(4).to(device)` — then read `.device` on the result.

In [ ]:
# Your turn!


# ---- Sample solution ----
z = torch.zeros(4).to(device)
print("device:", z.device)     # -> cpu (or cuda/mps if you have a GPU)

## 8. Autograd: gradients computed for you

This is PyTorch's headline feature. In notebook 03 you computed a gradient *by hand* to know which way to nudge a parameter. PyTorch does that automatically for **any** expression you build.

The recipe:
1. Mark the tensors you want gradients for with `requires_grad=True`.
2. Build an expression (a *forward* computation) that ends in a single number.
3. Call `.backward()` on that number.
4. Read each input's `.grad` — PyTorch filled it in with the slope of the output with respect to that input.

In [ ]:
# A single parameter we want to learn about.
w = torch.tensor(3.0, requires_grad=True)   # "track gradients for me"

# Build an expression. Think of a loss that depends on w.
y = w ** 2 + 2 * w          # y = w^2 + 2w

# Ask PyTorch for the gradient of y with respect to w.
y.backward()

# The slope dy/dw = 2w + 2. At w=3 that's 2*3 + 2 = 8.
print("w.grad:", w.grad)     # -> tensor(8.)

**What this does:** By flipping on `requires_grad`, PyTorch quietly records every operation you apply to `w` (it builds a little graph behind the scenes). `.backward()` walks that graph backward and computes the derivative, storing it in `w.grad`. This is *automatic differentiation* — the engine that lets us train models with millions of parameters without ever doing calculus by hand. During fine-tuning, `loss.backward()` fills in a `.grad` for every weight in the model at once.

### ✏️ Exercise

**Task:** Create `x = torch.tensor(4.0, requires_grad=True)`, build `z = 3 * x ** 2` (so `z = 3x²`), call `.backward()`, and print `x.grad`. The slope of `3x²` is `6x`, so at `x=4` you should get `24`.

**Hint:** Build the expression, call `z.backward()`, then read `x.grad`.

In [ ]:
# Your turn!


# ---- Sample solution ----
x = torch.tensor(4.0, requires_grad=True)
z = 3 * x ** 2
z.backward()
print("x.grad:", x.grad)     # -> tensor(24.)  (because 6*4 = 24)

## 9. `nn.Module` and `nn.Linear`: building a model

A PyTorch model is a class that inherits from `nn.Module`. You define two things:

- `__init__`: create the layers the model will use (here, one `nn.Linear`).
- `forward`: describe how an input flows through those layers to an output.

An `nn.Linear(in_features, out_features)` is the workhorse layer: it holds a weight matrix and a bias, and computes `input @ weight.T + bias`. You then *call* the model like a function — `model(x)` — and PyTorch runs your `forward` for you. **This is exactly what a Hugging Face model is, just with hundreds of layers instead of one.**

In [ ]:
class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()                 # always call this first
        # One layer: takes 3 numbers in, produces 1 number out.
        self.linear = nn.Linear(in_features=3, out_features=1)

    def forward(self, x):
        return self.linear(x)              # run the input through the layer

model = TinyModel()
print(model)                               # -> prints the layer structure

# The model's learnable numbers (weights + bias) live in .parameters().
for name, p in model.named_parameters():
    print(name, "->", tuple(p.shape))
# -> linear.weight -> (1, 3)
# -> linear.bias   -> (1,)

# Call the model on one example with 3 features. Output is 1 number.
x = torch.tensor([[1.0, 2.0, 3.0]])        # shape (1, 3): one row, 3 features
out = model(x)
print("model output:", out)                # -> tensor([[...]]) (value varies; weights are random)

**What this does:** `super().__init__()` wires up the `nn.Module` machinery (so PyTorch can find your parameters and gradients automatically). `nn.Linear(3, 1)` creates a weight of shape `(1, 3)` and a bias of shape `(1,)` — those are the numbers training will adjust. Calling `model(x)` triggers `forward`. The exact output is random right now because the weights start random; *training* is the process of changing them to be useful.

### ✏️ Exercise

**Task:** Define a model `Wider` whose single `nn.Linear` takes **4** features in and produces **2** numbers out. Create it, then feed it one example `torch.randn(1, 4)` and print the output's shape. What shape should the output be?

**Hint:** Copy `TinyModel` but change the `nn.Linear` to `nn.Linear(4, 2)`. With one input row, the output shape will be `(1, 2)`.

In [ ]:
# Your turn!


# ---- Sample solution ----
class Wider(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 2)

    def forward(self, x):
        return self.linear(x)

w_model = Wider()
example = torch.randn(1, 4)
print("output shape:", w_model(example).shape)   # -> torch.Size([1, 2])

## 10. Loss functions: measuring "how wrong"

A **loss** is a single number that says how far the model's prediction is from the truth — smaller is better. Two you'll use constantly:

- **`nn.MSELoss`** (Mean Squared Error) — for **regression** (predicting a number). It expects `prediction` and `target` to be the **same shape** and compares them directly.
- **`nn.CrossEntropyLoss`** — for **classification** (predicting a category). It expects raw scores (**logits**) of shape `(batch, num_classes)` and integer **class-index** targets of shape `(batch,)`. Note: it does *not* want one-hot vectors, and it expects logits, not probabilities — it applies the softmax internally.

In [ ]:
# --- MSE (regression): prediction and target are the same shape ---
mse = nn.MSELoss()
pred   = torch.tensor([2.5, 0.0, 2.0])
target = torch.tensor([3.0, 0.0, 1.0])
print("MSE loss:", mse(pred, target))   # -> tensor(0.4167) (mean of squared diffs)

# --- CrossEntropy (classification) ---
ce = nn.CrossEntropyLoss()
# 2 examples, 3 possible classes -> logits shape (2, 3).
logits = torch.tensor([[2.0, 0.5, 0.1],    # example 0: highest score on class 0
                       [0.1, 0.2, 3.0]])   # example 1: highest score on class 2
# True class index for each example (NOT one-hot): example 0 is class 0, example 1 is class 2.
labels = torch.tensor([0, 2])
print("CrossEntropy loss:", ce(logits, labels))  # -> a small loss (predictions match labels)

**What this does:** A loss function turns "prediction vs. truth" into one number you can minimize. The shapes are the gotcha: `MSELoss` wants matching shapes, while `CrossEntropyLoss` wants `(batch, num_classes)` logits paired with a `(batch,)` vector of integer class indices. The vast majority of fine-tuning classification jobs use `CrossEntropyLoss` exactly this way — and a wrong target shape here is one of the most common beginner errors.

### ✏️ Exercise

**Task:** Using `nn.MSELoss`, compute the loss between a prediction `[1.0, 2.0, 3.0]` and a target `[1.0, 2.0, 5.0]`. Only the last value differs (by 2). Print the loss.

**Hint:** The squared differences are `0, 0, 4`; their mean is the loss. Build an `nn.MSELoss()` and call it on the two tensors.

In [ ]:
# Your turn!


# ---- Sample solution ----
mse = nn.MSELoss()
pred   = torch.tensor([1.0, 2.0, 3.0])
target = torch.tensor([1.0, 2.0, 5.0])
print("MSE loss:", mse(pred, target))   # -> tensor(1.3333)  (mean of 0, 0, 4)

## 11. Optimizers: the rule for nudging weights

Autograd gives you each weight's gradient (which way is "uphill" on the loss). An **optimizer** uses those gradients to actually update the weights (step "downhill"). You give it the model's `.parameters()` and a **learning rate** (`lr`) — the step size.

- **`SGD`** (Stochastic Gradient Descent) — the simplest rule: `weight = weight - lr * gradient`.
- **`Adam`** — a smarter, adaptive version that usually trains faster and is the default for most deep learning. Fine-tuning typically uses **`AdamW`**, a close cousin.

Every training step follows the same three-beat rhythm:

1. `optimizer.zero_grad()` — clear old gradients (they otherwise *add up*).
2. `loss.backward()` — compute fresh gradients.
3. `optimizer.step()` — nudge every weight.

(The full loop is notebook 05; here's just a preview of the pieces.)

In [ ]:
model = TinyModel()

# Two ways to build an optimizer — pick one.
sgd  = torch.optim.SGD(model.parameters(),  lr=0.01)
adam = torch.optim.Adam(model.parameters(), lr=0.01)

print("SGD optimizer ready:", type(sgd).__name__)    # -> SGD
print("Adam optimizer ready:", type(adam).__name__)  # -> Adam

# The three-beat rhythm (one fake step shown):
x = torch.tensor([[1.0, 2.0, 3.0]])
target = torch.tensor([[10.0]])
loss = nn.MSELoss()(model(x), target)

adam.zero_grad()   # 1. clear old gradients
loss.backward()    # 2. compute new gradients
adam.step()        # 3. update the weights
print("did one optimizer step; loss was:", loss.item())  # .item() pulls out the plain number

**What this does:** The optimizer owns the *update rule*. You hand it `model.parameters()` once, and from then on `optimizer.step()` moves every weight a little, guided by the gradients that `backward()` produced. `zero_grad()` is easy to forget — without it, gradients accumulate across steps and training goes haywire. `.item()` converts a one-element tensor into an ordinary Python float for printing.

### ✏️ Exercise

**Task:** Create a fresh `TinyModel` and build an `SGD` optimizer for it with a learning rate of `0.1`. Print the learning rate by reading `optimizer.param_groups[0]["lr"]`.

**Hint:** `torch.optim.SGD(model.parameters(), lr=0.1)`. The current learning rate lives in `optimizer.param_groups[0]["lr"]`.

In [ ]:
# Your turn!


# ---- Sample solution ----
model = TinyModel()
opt = torch.optim.SGD(model.parameters(), lr=0.1)
print("learning rate:", opt.param_groups[0]["lr"])   # -> 0.1

## 12. Hands-on: tiny linear regression (watch the loss drop)

Time to snap all the pieces together. We'll make some data that roughly follows `y = 2x + 1`, give a model the job of discovering those numbers, and run a few training steps. The whole point: **watch the loss decrease** as the optimizer improves the weights. This is fine-tuning in miniature.

In [ ]:
torch.manual_seed(0)

# 1. Make data: y is about 2*x + 1, with a little noise.
x = torch.linspace(-1, 1, steps=20).reshape(-1, 1)   # shape (20, 1): 20 examples, 1 feature
y = 2 * x + 1 + 0.1 * torch.randn(x.shape)           # the "truth" we hope the model finds

# 2. Model: a single Linear(1 -> 1). It must learn weight≈2 and bias≈1.
model = nn.Linear(1, 1)

# 3. Loss + optimizer.
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# 4. Train for a handful of steps and print the loss each time.
for step in range(20):
    pred = model(x)              # forward pass
    loss = loss_fn(pred, y)      # how wrong?

    optimizer.zero_grad()        # clear old gradients
    loss.backward()              # compute new gradients
    optimizer.step()             # nudge the weights downhill

    if step % 5 == 0:
        print(f"step {step:2d}  loss = {loss.item():.4f}")
# -> loss starts higher and DROPS each printed step (exact numbers vary by run)

# 5. Did it recover y = 2x + 1?
w = model.weight.item()
b = model.bias.item()
print(f"learned weight = {w:.2f} (target ~2.0), bias = {b:.2f} (target ~1.0)")

**What this does:** This is a complete (if tiny) training run. Each loop iteration does a forward pass, measures the loss, and runs the `zero_grad → backward → step` rhythm to improve the weights. After 20 steps the learned `weight` and `bias` should be close to 2 and 1 — the model *discovered* the rule behind the data. A real fine-tune is this same loop, just with a giant model, a real dataset, and `CrossEntropyLoss`.

### ✏️ Exercise

**Task:** Re-run the training loop above but change the learning rate from `0.1` to `0.5`. Does the loss drop faster, slower, or does it become unstable (jump around / grow)? Try `0.01` too. Write a one-line comment with what you observed.

**Hint:** Just edit the `lr=` value in the optimizer and re-run. A too-small `lr` learns slowly; a too-large `lr` can overshoot and make the loss *increase*. Finding a good learning rate is one of the most important fine-tuning skills.

In [ ]:
# Your turn! Copy the loop, change lr, and compare.


# ---- Sample solution ----
def train(lr, steps=20):
    torch.manual_seed(0)
    x = torch.linspace(-1, 1, steps=20).reshape(-1, 1)
    y = 2 * x + 1 + 0.1 * torch.randn(x.shape)
    model = nn.Linear(1, 1)
    loss_fn = nn.MSELoss()
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    for _ in range(steps):
        loss = loss_fn(model(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return loss.item()

for lr in [0.01, 0.1, 0.5]:
    print(f"lr={lr}: final loss = {train(lr):.4f}")
# Observation (typical): lr=0.5 learns fastest here; lr=0.01 is much slower.
# Push lr far higher (e.g. 5.0) and the loss would blow up instead.

## Common mistakes & how to debug them

- **Shape mismatch in matrix multiply.** Error like *"mat1 and mat2 shapes cannot be multiplied (2x3 and 2x2)"*. Print both shapes and check the **inner** dimensions match: `(m, k) @ (k, n)`. Reshape with `view`/`reshape` or add/remove a batch dimension as needed.

- **dtype mismatch.** *"expected scalar type Float but found Long"* means you handed an int tensor where a float was wanted. Fix with `.float()`. (Class-index labels for `CrossEntropyLoss`, though, must stay integer / `long`.)

- **Wrong target shape for the loss.** `MSELoss` wants prediction and target to be the **same shape**. `CrossEntropyLoss` wants `(batch, num_classes)` logits and a `(batch,)` vector of integer class indices — *not* one-hot vectors and *not* probabilities.

- **Forgetting `optimizer.zero_grad()`.** Gradients **accumulate** by default. Skip the reset and each step's gradient piles onto the last, and training diverges. Always `zero_grad()` before `backward()`.

- **Device mismatch.** *"Expected all tensors to be on the same device"* means some data is on CPU and some on GPU. Move everything with `.to(device)` — model *and* inputs *and* targets.

- **Calling `.forward()` directly.** Use `model(x)`, not `model.forward(x)`. The call form runs extra PyTorch bookkeeping that `forward` alone skips.

- **Reading `.grad` before `.backward()`.** It'll be `None`. Gradients only exist after a backward pass, and only for tensors with `requires_grad=True`.

## Summary

- **Tensors** are number grids with a `.shape` and a `.dtype`; create them with `tensor`, `zeros`, `ones`, `randn`, `arange`, slice them like lists, and reshape with `view`/`reshape` (`-1` infers a dimension).
- They **bridge to NumPy** via `.numpy()` and `torch.from_numpy` (which can share memory on CPU).
- **Math** is mostly elementwise; `@` is matrix multiply (the core of a `Linear` layer); **broadcasting** stretches compatible shapes; `sum`/`mean`/`argmax` reduce.
- **Devices** (`cpu`/`cuda`/`mps`) are detected with a fallback ladder; `.to(device)` moves tensors and models, and everything must share a device.
- **Autograd** computes gradients automatically: set `requires_grad=True`, call `.backward()`, read `.grad` — the auto version of notebook 03's hand-calculus.
- A model is an **`nn.Module`** with `__init__` + `forward`; `nn.Linear` is the workhorse layer; you call it as `model(x)`. **Every Hugging Face model is one of these.**
- **Losses** measure wrongness: `MSELoss` (same-shape regression) and `CrossEntropyLoss` (logits + integer labels for classification).
- **Optimizers** (`SGD`, `Adam`) apply the `zero_grad → backward → step` rhythm to nudge weights — and you watched a tiny model's loss actually drop.

## What to learn next

Next up: **`05_training_loop_from_scratch.ipynb`**, where we take the `zero_grad → backward → step` rhythm you just previewed and build a *complete, honest training loop* — epochs, batches, tracking the loss, and evaluating on held-out data. Everything in this notebook (tensors, autograd, `nn.Module`, losses, optimizers) becomes a moving part of that loop, which is the very same loop the Hugging Face `Trainer` runs for you under the hood.

See you in notebook 05!